# Linking neural manifolds to circuit structure in recurrent networks

>Reference: **Pezon, Schmutz & Gerstner (2026), *[Linking neural manifolds to circuit structure in recurrent networks](https://doi.org/10.1016/j.neuron.2025.12.047)*, Neuron**.

**Abstract**: Two of the most successful ways of making sense of
population recordings are both forms of dimensionality reduction, applied to opposite
sides of the activity matrix. Reduce across *time* and each neuron becomes a point in a
"similarity space" (this is what connectomics-style analyses, t-SNE of tuning curves, or
PCA loadings give you). Reduce across *neurons* and each moment becomes a point on a
"neural manifold" (this is what state-space trajectories give you). Both are routinely
measured, but neither tells you what the underlying *circuit* looks like — where neurons
sit in some functional space and how their positions determine their connections. This
paper asks: if we model the circuit as neurons placed at locations in a low-dimensional
"circuit space" with connectivity a smooth function of location, what can the observed
activity — the manifold, or the neuronal embeddings — tell us about that space? The
answers are sobering and useful at once: the latent dynamics alone *cannot* pin down the
circuit (very different circuits produce identical dynamics), but the *geometry of the
neuronal embeddings* leaves footprints — symmetries and, most powerfully, topology — that
rule out whole families of circuits.


| Section | Content | Paper figure |
|---|---|---|
| Background | two views of population activity, and the missing link to circuits | Fig. 1 |
| The framework | circuit space, low-rank connectivity, latent dynamics | Fig. 2 |
| A 1D circuit with 2D dynamics | piecewise weights on a Gaussian line, three attractors | Fig. 3B |
| Different circuits, identical dynamics | ring vs planar circuit, one shared limit cycle | Fig. 3C–D |
| The task | context-dependent decision-making | Fig. 4A |
| Five circuits, one solution | aligned vs orthogonal inputs | Fig. 4B |
| Simulation | choices read out from the population | Fig. 4D |
| Same latent dynamics | context-dependent fields and TDR trajectories | Fig. 4C, E |
| Different neuronal embeddings | what PCA loadings reveal | Fig. 4F |
| The discrimination table | topological footprints rule circuits in or out | Fig. 4H |

## Setup

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

MODEL_DIR = Path("models/manifold_and_circuit")   # cached simulations
FIG_DIR = Path("figs/manifold_and_circuit")       # all figures land here
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps


def set_rcparams():
    """Journal-style defaults used by all figures of this notebook."""
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
        "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
        "axes.linewidth": 0.8, "lines.linewidth": 1.2,
        "axes.spines.top": False, "axes.spines.right": False,
    })


# --------------------------------------------------------------------- #
# Vector fields in the latent plane
# --------------------------------------------------------------------- #
def plot_latent_field(ax, field_fn, lim, n_grid=25, n_stream=40,
                      cmap="viridis", seed_points=None, extra_args=()):
    """Speed heatmap + streamlines of a 2D latent field on [-lim, lim]^2.

    field_fn takes a length-2 kappa and returns d kappa / dt. Returns the
    speed grid (useful for a shared colorbar).
    """
    grid = np.linspace(-lim, lim, n_grid)
    K1, K2 = np.meshgrid(grid, grid)
    speed = np.zeros_like(K1)
    for i in range(n_grid):
        for j in range(n_grid):
            v = field_fn(np.array([K1[i, j], K2[i, j]]), *extra_args)
            speed[i, j] = np.linalg.norm(v)
    ax.pcolormesh(K1, K2, speed, cmap=cmap, shading="auto", alpha=0.85)
    U = np.zeros_like(K1)
    V = np.zeros_like(K2)
    for i in range(n_grid):
        for j in range(n_grid):
            U[i, j], V[i, j] = field_fn(np.array([K1[i, j], K2[i, j]]),
                                        *extra_args)
    ax.streamplot(K1, K2, U, V, color="w", density=n_stream / 15,
                  linewidth=0.6, arrowsize=0.7)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    return speed


def plot_fixed_points(ax, fps, unit, stable=("fp1", "fp2", "fp3"),
                      saddles=("saddle1", "saddle2")):
    """Mark stable fixed points (filled circles) and saddles (crosses)."""
    for name in stable:
        ax.plot(fps[name][0] / unit, fps[name][1] / unit, "o", ms=7,
                mfc="none", mec="crimson", mew=1.5)
    for name in saddles:
        ax.plot(fps[name][0] / unit, fps[name][1] / unit, "x", ms=7,
                mec="crimson", mew=1.5)


# --------------------------------------------------------------------- #
# Embeddings (PCA loadings = similarity-space coordinates of neurons)
# --------------------------------------------------------------------- #
def plot_embedding_2d(ax, loadings, color_values, cmap="viridis", s=3,
                      colorbar_label=None):
    """Scatter of the first two PC loadings, one point per neuron, colored
    by a circuit-space coordinate."""
    sc = ax.scatter(loadings[:, 0], loadings[:, 1], c=color_values, cmap=cmap,
                    s=s, rasterized=True)
    ax.set_xlabel("PC 1 loading")
    ax.set_ylabel("PC 2 loading")
    ax.set_aspect("equal")
    if colorbar_label:
        plt.colorbar(sc, ax=ax, label=colorbar_label)
    return sc


def plot_embedding_3d(ax, loadings, color_values, cmap="viridis", s=2,
                      shadow_offset=None, elev=20, azim=-60):
    """3D scatter of the first three PC loadings with gray 2D shadows on
    the three coordinate planes (the shadow trick makes the point-cloud
    geometry readable in print)."""
    lim = 1.1 * np.abs(loadings[:, :3]).max()
    off = shadow_offset if shadow_offset is not None else lim
    for proj, fixed, zdir in [((0, 1), 2, "z"), ((0, 2), 1, "y"),
                              ((1, 2), 0, "x")]:
        pts = loadings[:, list(proj)]
        ax.scatter(pts[:, 0], pts[:, 1], zs=-off, zdir=zdir,
                   c="0.75", s=0.5, rasterized=True, depthshade=False)
    sc = ax.scatter(loadings[:, 0], loadings[:, 1], loadings[:, 2],
                    c=color_values, cmap=cmap, s=s, rasterized=True,
                    depthshade=False)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.set_xlabel("PC 1", labelpad=-8)
    ax.set_ylabel("PC 2", labelpad=-8)
    ax.set_zlabel("PC 3", labelpad=-8)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    return sc


# --------------------------------------------------------------------- #
# Task and readout
# --------------------------------------------------------------------- #
def plot_input_timeline(ax, times, inputs, t_max, labels, colors):
    """The four latent input channels over the first trials of the task,
    each normalized to unit amplitude and stacked vertically."""
    sel = times <= t_max
    for k, (lab, c) in enumerate(zip(labels, colors)):
        trace = inputs[sel, k]
        trace = trace / (np.abs(trace).max() + 1e-12)
        ax.plot(times[sel], trace + (len(labels) - 1 - k) * 2.2, color=c,
                lw=0.8)
        ax.text(times[sel][-1] * 1.01, (len(labels) - 1 - k) * 2.2, lab,
                color=c, va="center", fontsize=7)
    ax.set_yticks([])
    ax.set_xlim(0, t_max * 1.16)
    ax.set_xlabel("time (ms)")
    ax.set_ylabel("input (normalized)")


def plot_choice_traces(ax, times_stored, choice_trace, task, n_trials_show=8,
                       t_trial=500.0):
    """Choice readout over the first trials, colored by the correct choice,
    with shaded context windows."""
    t_max = n_trials_show * t_trial
    sel = times_stored <= t_max
    ax.axhline(0, color="0.5", lw=0.5, zorder=0)
    ax.plot(times_stored[sel], choice_trace[sel], color="k", lw=0.7)
    steps = int(round(t_trial / (times_stored[1] - times_stored[0])))
    correct = np.where(task["conts"] > 0, task["s1"], task["s2"])
    for n in range(n_trials_show):
        tr = choice_trace[n * steps:(n + 1) * steps]
        sign = np.sign(tr[np.argmax(np.abs(tr))])
        color = "tab:green" if sign == correct[n] else "crimson"
        ax.plot(times_stored[n * steps:(n + 1) * steps], tr, color=color,
                lw=0.9)
    ax.set_xlabel("time (ms)")
    ax.set_ylabel("choice readout (a.u.)")


# --------------------------------------------------------------------- #
# TDR trajectories
# --------------------------------------------------------------------- #
def plot_tdr_panel(ax, trajs_avg, task, dt_stored, context_value,
                   model_colors, model_names, t_trial=500.0,
                   arrow_step=100):
    """Trial-averaged latent trajectories in the color-motion TDR plane for
    one context, one line per (model, condition). trajs_avg: dict
    model -> (4 conditions x 2 signs... arranged as (8, 2, T)) projections
    already block-averaged; rows ordered by (s1, s2, conts) triples.
    """
    steps = trajs_avg[model_names[0]].shape[-1]
    t_in_trial = np.arange(steps) * dt_stored
    for name in model_names:
        tr = trajs_avg[name]           # (n_trials, 2, steps)
        for n in range(tr.shape[0]):
            if task["conts"][n] != context_value:
                continue
            x, y = tr[n, 0], tr[n, 1]
            ax.plot(x, y, color=model_colors[name], lw=0.9, alpha=0.8)
            # direction arrow mid-trial
            i0 = int(200 / dt_stored)
            if i0 + 2 < steps:
                ax.annotate("", xy=(x[i0 + 2], y[i0 + 2]),
                            xytext=(x[i0], y[i0]),
                            arrowprops=dict(arrowstyle="->",
                                            color=model_colors[name],
                                            lw=0.9))
    ax.axhline(0, color="0.7", lw=0.5)
    ax.axvline(0, color="0.7", lw=0.5)
    ax.set_xlabel("color axis (a.u.)")
    ax.set_ylabel("motion axis (a.u.)")


# --------------------------------------------------------------------- #
# Discrimination table (topological footprints)
# --------------------------------------------------------------------- #
TOPOLOGY = {   # model -> (intrinsic dim of Z, connected components of Z)
    "ring": (1, 1), "MS": (2, 1), "PS": (1, 1), "clust": (0, 4),
    "hid": (1, 2),
}


def discrimination_matrix(model_order):
    """excluded[i, j] is True when the circuit of model j cannot generate
    the neuronal embedding observed in the data of model i: the embedding
    cannot have higher intrinsic dimension or more connected components
    than the circuit space that generated it (topological footprint)."""
    n = len(model_order)
    excluded = np.zeros((n, n), dtype=bool)
    for i, data in enumerate(model_order):
        d_emb, c_emb = TOPOLOGY[data]
        for j, model in enumerate(model_order):
            d_z, c_z = TOPOLOGY[model]
            excluded[i, j] = (d_emb > d_z) or (c_emb > c_z)
    return excluded


def plot_discrimination_table(ax, model_order, excluded, labels=None):
    """Render the exclusion matrix as a grid: dark cross = excluded."""
    n = len(model_order)
    labels = labels or model_order
    for i in range(n):
        for j in range(n):
            if excluded[i, j]:
                ax.plot(j, n - 1 - i, marker="x", ms=10, mew=2.0,
                        color="crimson")
            else:
                ax.plot(j, n - 1 - i, marker="o", ms=6, mfc="none",
                        mec="0.4", mew=1.0)
    ax.set_xticks(range(n), labels)
    ax.set_yticks(range(n), labels[::-1])
    ax.set_xlabel("candidate circuit")
    ax.set_ylabel("observed embedding (data)")
    ax.set_xlim(-0.6, n - 0.4)
    ax.set_ylim(-0.6, n - 0.4)
    for s in ax.spines.values():
        s.set_visible(True)
    ax.set_aspect("equal")

set_rcparams()

## Background

**Two ways to compress a population recording.** An experiment gives an activity matrix:
neurons × time (or trials). Averaging or reducing along the time axis says which *neurons*
behave alike — each neuron becomes a point in a similarity space, and clusters or
continua in that space are read as cell types or functional classes. Reducing along the
neuron axis says which *moments* behave alike — the population state traces a
low-dimensional manifold, and computations are read off its trajectories, fixed points,
and transitions. Both views are staples of modern systems neuroscience, and both are
purely descriptive of *activity*.

**What neither view tells you.** Activity is generated by a circuit: neurons wired
together by synapses whose strengths depend on what the neurons are. Structured models of
such circuits — from ring models of orientation tuning to clustered networks — place
neurons at locations $z_i$ in a low-dimensional *circuit space* $\mathcal{Z}$ (angle on a
ring, position in a feature plane, a cluster label, ...) and let the connection from $j$
to $i$ be a smooth function of the two locations. The circuit space is the object a
connectomic or functional-clustering analysis would love to recover. The question of this
paper is sharp: **which properties of $\mathcal{Z}$ are footprints in observable
activity, and which are forever undecidable from activity alone?**

**Why this matters to a cognitive neuroscientist.** If you have ever plotted a state-space
trajectory and wondered whether it constrains the wiring, the answer here is: much less
than one hopes (different wirings give identical trajectories), but the *neuronal
embeddings* — the point cloud neurons form when each is embedded by its activity pattern —
carry hard topological constraints. An embedding cannot have higher intrinsic dimension
or more connected components than the circuit space that generated it. That single rule,
applied to a decision-making network, is enough to discriminate five circuits that all
solve the task identically.

## The framework: circuit space, latent dynamics, and the three kinds of footprint

**Structured circuit models.** Place $N$ neurons at locations $z_i$ drawn from a
distribution $\rho$ on a circuit space $\mathcal{Z}$ of intrinsic dimension $d$. The
recurrent weight from $j$ to $i$ is a dot product of $D$-dimensional connectivity
vectors,

$$W_{ij} = f(z_i)\cdot g(z_j) = \sum_{\mu=1}^D f_\mu(z_i)\, g_\mu(z_j),$$

i.e. $W = F G^{T}$ (rank $D$), scaled by $1/N$ so that total synaptic input stays finite
as $N$ grows. Neuron $i$ integrates its input with time constant $\tau$,

$$\dot h_i = -\frac{h_i}{\tau} + \frac{1}{N}\sum_j W_{ij}\,\varphi(h_j) + \text{inputs},$$

with transfer function $\varphi(h) = \tfrac{R}{2}\bigl(1 + \mathrm{erf}(h-\theta)\bigr)$ —
a smooth sigmoid bounded in $[0, R]$.

**Latent dynamics.** Because $W$ is low-rank, the membrane potentials are exactly
parametrized by $D$ latent variables $\kappa(t)$: $h_i(t) = f(z_i)\cdot \kappa(t)$. In
the large-$N$ limit the sum over neurons becomes an integral over circuit space, giving a
closed $D$-dimensional dynamical system,

$$\frac{d\kappa_\mu}{dt} = -\frac{\kappa_\mu}{\tau} + \bigl\langle g_\mu,\,
\varphi(f\cdot\kappa)\bigr\rangle_{\rho} + I_\mu(t).$$

The full $N$-dimensional network and this small system are *the same dynamics*, viewed at
two resolutions. Along any simulation we track the latents by the least-squares readout
$\kappa = (F^{T}F/N)^{-1} F^{T} h / N$.

**Three results that organize everything below.**

* **Degeneracy.** The latent dynamics sees the connectivity functions only through the
  averages $\langle g_\mu, \varphi(f\cdot\kappa)\rangle_\rho$, and many different pairs
  $(\rho, f, g)$ give the same averages. The circuit-space dimension $d$ can differ from
  the dynamics dimension $D$, and macroscopically identical dynamics can come from
  microscopically different circuits. Latent trajectories alone cannot identify a circuit.
* **Symmetry footprints.** A microscopic symmetry (permuting neurons by a transformation
  of $\mathcal{Z}$ that leaves the circuit invariant) induces a symmetry of the latent
  vector field. A visible symmetry of the dynamics therefore certifies a corresponding
  symmetry of the circuit — a one-way street.
* **Topological footprints.** Each neuron, embedded by its activity (e.g. its PCA
  loadings), is a smooth image of its circuit location. A continuous map cannot raise
  intrinsic dimension or merge disconnected pieces, so the neuronal embedding cannot have
  higher intrinsic dimension or more connected components than $\mathcal{Z}$. Violations
  rule a candidate circuit out, full stop.

**A note on the simulations.** The paper derives this theory for spiking networks and
simulates Poisson neurons; here every network is simulated as the deterministic **rate
model** $\dot h = -h/\tau + W\varphi(h)/N + F\kappa_I$ — the large-$N$ limit in which
spike noise vanishes, which is the theoretically meaningful object. External input noise
(the task inputs below carry deliberate noise) keeps single-trial activity variable, so
all embedding analyses behave as in the spiking version. Euler steps are in milliseconds,
as in the theory.

### The simulator

In [ ]:
import numpy as np
import torch


def erf_transfer(x, R=1.0, threshold=0.0):
    """Voltage-to-rate transfer function phi(h) = R (1 + erf(h - theta)) / 2.

    A smooth sigmoid bounded in [0, R]; R is the maximal firing rate in kHz
    (spikes per ms) and theta a soft threshold. Smoothness of phi is what
    guarantees that nearby circuit locations imply nearby activities.
    """
    return R * (1.0 + torch.erf(x - threshold)) / 2.0


def connectivity_matrices(Z, f_funcs, g_funcs):
    """Evaluate the connectivity functions on sampled locations.

    Z: (N, d) array of neuron locations in circuit space; f_funcs, g_funcs:
    lists of D callables taking Z and returning length-N vectors.
    Returns F, G of shape (N, D).
    """
    F = np.stack([f(Z) for f in f_funcs], axis=1)
    G = np.stack([g(Z) for g in g_funcs], axis=1)
    return F, G


def latent_projection(F):
    """Matrix M = (F F^T / N)^-1 mapping membrane potentials to latents."""
    N = F.shape[0]
    return np.linalg.inv(F.T @ F / N)


@torch.no_grad()
def run_network(F, G, kappa0, times, tau, phi, kappa_I=None, store_every=1,
                device="cpu", dtype=torch.float32):
    """Euler-integrate the rate dynamics and track the latent variables.

    F, G: (N, D) connectivity matrices; kappa0: (D,) initial latent state
    (the initial membrane potentials are h0 = F kappa0); times: (T,) time
    points in ms (uniform step dt = times[1] - times[0]); tau: membrane
    time constant in ms; phi: transfer function on torch tensors;
    kappa_I: optional (T, D) latent-space external input, added to the
    recurrent drive before multiplication by F (aligned inputs occupy
    columns whose g is nonzero; an input column with g = 0 is the
    "orthogonal" input of the theory, entering only inside phi);
    store_every: keep one time slice out of this many for the rates.

    Returns dict with h_t and rates r_t = phi(h_t) at the stored times,
    and the latent trajectory kappa_t at full resolution.
    """
    F_t = torch.as_tensor(F, dtype=dtype, device=device)
    G_t = torch.as_tensor(G, dtype=dtype, device=device)
    N, D = F_t.shape
    M_t = torch.as_tensor(latent_projection(np.asarray(F, dtype=np.float64)),
                          dtype=dtype, device=device)

    T = len(times)
    dt = float(times[1] - times[0])
    if kappa_I is None:
        kappa_I = np.zeros((T, D))
    kI_t = torch.as_tensor(kappa_I, dtype=dtype, device=device)

    h = F_t @ torch.as_tensor(kappa0, dtype=dtype, device=device)
    store_idx = np.arange(0, T, store_every)
    h_t = torch.zeros(len(store_idx), N, dtype=dtype, device=device)
    kappa_t = torch.zeros(T, D, dtype=dtype, device=device)

    for t in range(T):
        if t % store_every == 0:
            h_t[t // store_every] = h
        kappa_t[t] = M_t @ (F_t.T @ h) / N
        drive = (G_t.T @ phi(h)) / N + kI_t[t]     # latent-space drive, D-dim
        h = (1.0 - dt / tau) * h + (F_t @ drive) * dt

    return {
        "h_t": h_t.cpu().numpy(),
        "r_t": phi(h_t).cpu().numpy(),
        "kappa_t": kappa_t.cpu().numpy(),
        "times_stored": np.asarray(times)[store_idx],
    }

### Analysis tools

Two ingredients shared by everything below: a causal low-pass filter and PCA of the
activity matrix (via the framework's `fit_pca`) for neuronal embeddings and denoising;
readout axes built from the connectivity vectors; and targeted dimensionality reduction
(TDR) for the decision task.

In [ ]:
import numpy as np

from neuralrnn.analysis import fit_pca


def low_pass(x, alpha=1 / 50):
    """Causal exponential filter y_t = (1 - alpha) y_{t-1} + alpha x_t along
    axis 0; alpha = dt / tau_filter (here 1 ms steps, tau_filter = 50 ms)."""
    y = np.empty_like(x, dtype=np.float64)
    y[0] = x[0]
    for t in range(1, len(x)):
        y[t] = (1.0 - alpha) * y[t - 1] + alpha * x[t]
    return y


def pca_fit(activity, n_components=7):
    """PCA on (T, N) activity. Returns components (k, N) (the PC axes;
    each neuron's coordinates on them are its similarity-space embedding),
    the mean, and the explained variance ratios."""
    res = fit_pca(np.asarray(activity, dtype=np.float64), n_components)
    return {"components": res.components, "mean": res.mean,
            "ratios": res.explained_variance_ratio}


def pca_denoise(pca, activity):
    """Reconstruct activity from the fitted PCs (removes noise orthogonal
    to the first components)."""
    V = pca["components"]
    Xc = activity - pca["mean"]
    return (Xc @ V.T) @ V + pca["mean"]


def frac_var_explained(axes, activity):
    """Variance captured by each axis (columns of axes, N x k) as a
    fraction of the total variance of the (T, N) activity."""
    axs = axes / np.linalg.norm(axes, axis=0, keepdims=True)
    total_var = activity.var(axis=0, ddof=1).sum()
    return np.array([activity @ axs[:, i] for i in range(axs.shape[1])]
                    ).var(axis=1, ddof=1) / total_var


# --------------------------------------------------------------------- #
# Readouts
# --------------------------------------------------------------------- #
def readout_axes(F, unit_kappa, renorm=1e4):
    """Normalized projection axes from the connectivity columns f1, f2 and
    the choice axis f1 + f2 (F: (N, 4) connectivity matrix)."""
    F1, F2 = F[:, 0], F[:, 1]
    norm = lambda v: v / np.sum(v ** 2) / len(v) / unit_kappa * renorm
    a_col, a_mot = norm(F1), norm(F2)
    return {"color": a_col, "motion": a_mot, "choice": a_col + a_mot}


def trial_choices(choice_trace, times, t_trial=500.0):
    """One choice per trial: sign of the choice readout at the time of
    maximal absolute value within the trial."""
    steps = int(round(t_trial / (times[1] - times[0])))
    n_trials = len(times) // steps
    return np.array([
        np.sign(tr[np.argmax(np.abs(tr))])
        for tr in choice_trace[:n_trials * steps].reshape(n_trials, steps)])


def correct_choices(task):
    """The correct choice on each trial: sign of the relevant feature."""
    return np.where(task["conts"] > 0, task["s1"], task["s2"])


# --------------------------------------------------------------------- #
# Targeted dimensionality reduction
# --------------------------------------------------------------------- #
def tdr_axes(denoised, activity, task, t_trial=500.0, dt_stored=1.0):
    """Orthonormal color / motion / context / bias axes (N, 4) from the
    denoised (T, N) activity, with signs fixed against reference windows
    of the raw activity (trial 2 for color, trial 1 for motion/context)."""
    steps = int(round(t_trial / dt_stored))
    n_trials = len(denoised) // steps
    act_t = denoised[:n_trials * steps].reshape(n_trials, steps, -1)
    # act_t: (trial, time, neuron)

    design = np.stack([task["s1"], task["s2"], task["conts"],
                       np.ones(n_trials)])          # (4, n_trials)
    proj = np.linalg.inv(design @ design.T) @ design  # (4, n_trials)
    betas = np.einsum("fr,rtn->ftn", proj, act_t)     # (4, time, neuron)
    max_t = (betas ** 2).sum(axis=2).argmax(axis=1)   # per factor
    beta_max = np.stack([betas[f, max_t[f], :] for f in range(4)])  # (4, N)
    Q, _ = np.linalg.qr(beta_max.T)                   # (N, 4) orthonormal

    # fix signs so that color is positive late in trial 2 (a color+ trial),
    # motion and context positive in trial 1 (a motion+ / context- trial)
    T1 = slice(199, 300)
    T2 = slice(699, 800)
    signs = np.array([
        np.sign(Q[:, 0] @ activity[T2].mean(axis=0)),
        np.sign(Q[:, 1] @ activity[T1].mean(axis=0)),
        np.sign(Q[:, 2] @ activity[T1].mean(axis=0)),
        1.0])
    signs[signs == 0] = 1.0
    return Q * signs


def block_average(trajs, n_reps=4):
    """Average (k, T) trajectories over the n_reps repetitions of the
    8-condition block, returning (k, T / n_reps)."""
    k, T = trajs.shape
    return trajs.reshape(k, n_reps, T // n_reps).mean(axis=1)

## A one-dimensional circuit with two-dimensional dynamics

The first theory example shows that $d$ (circuit-space dimension) and $D$ (dynamics
dimension) can already differ in the simplest setting. Neurons live on a line,
$z \sim \mathcal{N}(0,1)$, so $d = 1$. Connectivity is rank-one in spirit but with
negative weights weakened by a factor $a = \cos\alpha$, $\alpha = \pi/3$:

$$W_{ij} = \begin{cases} z_i z_j & z_i z_j \ge 0\\ a\, z_i z_j & z_i z_j < 0\end{cases}$$

which is exactly rank $D = 2$: with $[z]_\pm$ the positive/negative parts and
$b = -\sin\alpha$,

$$f(z) = \bigl([z]_+ + a[z]_-,\; b\,[z]_-\bigr),\qquad g(z) = J f(z),\qquad J = 0.8.$$

Because $z$ is Gaussian, the circuit-space integral in the latent dynamics can be done in
closed form for the erf transfer function. With $\lambda = A^{T}\kappa$,
$A = \begin{pmatrix}1 & a\\ 0 & b\end{pmatrix}$ and
$\Phi(x) = \bigl(1 + x/\sqrt{x^2 + 1/2}\bigr)/\bigl(2\sqrt{2\pi}\bigr)$,

$$\frac{d\kappa}{dt} = -\frac{\kappa}{\tau} + J R\, A
\begin{pmatrix}\Phi(\lambda_1)\\ -\Phi(-\lambda_2)\end{pmatrix},$$

a planar field we can draw exactly, with axes in units of $k = J\tau R/\sqrt{2\pi}$. It
hosts **three stable fixed points and two saddles**. Simulated finite-$N$ networks should
fall onto the attractors of this analytic field.

In [ ]:
import numpy as np
from scipy.optimize import fsolve
from scipy.special import i0, i1

TAU = 10.0          # membrane time constant, ms
R_RATE = 1.0        # maximal firing rate, kHz
N_NEURONS = 20000   # neurons per network (see note below)
DT = 0.1            # Euler step, ms
T_SIM = 300.0       # simulation length, ms

# Note on N: with only 2000 sampled locations the sampled vector field of
# the deterministic rate model is distorted enough to visibly warp the
# limit cycle (~10% radius modulation for the planar circuit), because the
# radial dynamics is nearly marginal where the cycle sits. With N = 20000
# the sampled field is within a few percent of the large-N theory, which is
# what the figures illustrate. Nothing qualitative depends on this choice.


# --------------------------------------------------------------------- #
# Example 1: piecewise weights on a one-dimensional circuit
# --------------------------------------------------------------------- #
EX1 = dict(J=0.8, alpha=np.pi / 3)


def example1_functions(J=EX1["J"], alpha=EX1["alpha"]):
    """Connectivity functions f = (f1, f2), g = J f for Example 1."""
    a, b = np.cos(alpha), -np.sin(alpha)
    f1 = lambda Z: Z[:, 0] * ((Z[:, 0] > 0) + a * (Z[:, 0] < 0))
    f2 = lambda Z: Z[:, 0] * (b * (Z[:, 0] < 0))
    return [f1, f2], [lambda Z: J * f1(Z), lambda Z: J * f2(Z)]


def example1_field(kappa, J=EX1["J"], alpha=EX1["alpha"], tau=TAU, R=R_RATE):
    """Analytic latent vector field of Example 1 (Gaussian quadrature done
    in closed form for the erf transfer function)."""
    a, b = np.cos(alpha), -np.sin(alpha)
    A = np.array([[1.0, a], [0.0, b]])
    lam = A.T @ np.asarray(kappa, dtype=float)
    Phi = lambda x: (1.0 + x / np.sqrt(x ** 2 + 0.5)) / (2.0 * np.sqrt(2.0 * np.pi))
    drive = J * R * A @ np.array([Phi(lam[0]), -Phi(-lam[1])])
    return -np.asarray(kappa, dtype=float) / tau + drive


def example1_fixed_points(J=EX1["J"], alpha=EX1["alpha"]):
    """Three stable fixed points and two saddles of the Example-1 field,
    found from the standard initial guesses (in units of k = J tau R /
    sqrt(2 pi))."""
    k = example1_unit_kappa(J)
    guesses = {"saddle1": [0.0, 1.0], "saddle2": [1.0, 0.5],
               "fp1": [-0.5, 1.0], "fp2": [1.0, -0.5], "fp3": [1.0, 1.0]}
    return {name: fsolve(example1_field, k * np.array(g), args=(J, alpha))
            for name, g in guesses.items()}


def example1_unit_kappa(J=EX1["J"]):
    """Scale k = J tau R / sqrt(2 pi) used to normalize latent axes."""
    return J * TAU * R_RATE / np.sqrt(2.0 * np.pi)


# --------------------------------------------------------------------- #
# Example 2: ring vs planar circuit, identical limit cycle
# --------------------------------------------------------------------- #
EX2 = dict(delta=np.pi / 10, J=1.0, sigma=0.6, J_gaus=1.39, r0=2.9348)


def rot2d(delta):
    """2x2 rotation matrix by angle delta."""
    c, s = np.cos(delta), np.sin(delta)
    return np.array([[c, -s], [s, c]])


def example2_functions(model, delta=EX2["delta"], J=EX2["J"],
                       sigma=EX2["sigma"], J_gaus=EX2["J_gaus"]):
    """Connectivity functions for the ring (model="ring") or the planar
    (model="gaus") circuit of Example 2."""
    if model == "ring":
        f = [lambda Z: np.cos(Z[:, 0]), lambda Z: np.sin(Z[:, 0])]
        g = [lambda Z: J * np.cos(Z[:, 0] + delta),
             lambda Z: J * np.sin(Z[:, 0] + delta)]
    else:
        Rm = rot2d(delta)
        f = [lambda Z: Z[:, 0], lambda Z: Z[:, 1]]
        g = [lambda Z: J_gaus * (Z @ Rm.T)[:, 0],
             lambda Z: J_gaus * (Z @ Rm.T)[:, 1]]
    return f, g


def radial_F_ring(r, J=EX2["J"], tau=TAU, R=R_RATE):
    """Radial drive of the ring model: J R r / (2 sqrt(pi)) e^{-r^2/2}
    [I_1(r^2/2) + I_0(r^2/2)] with I_n the modified Bessel functions."""
    x = np.asarray(r, dtype=float)
    return J * R * x / (2.0 * np.sqrt(np.pi)) * np.exp(-x ** 2 / 2.0) \
        * (i1(x ** 2 / 2.0) + i0(x ** 2 / 2.0))


def radial_F_gaus(r, sigma=EX2["sigma"], J_gaus=EX2["J_gaus"], R=R_RATE):
    """Radial drive of the planar model: J' R r sigma^2 / sqrt(pi (1 + 2
    sigma^2 r^2))."""
    x = np.asarray(r, dtype=float)
    return J_gaus * R * x * sigma ** 2 / np.sqrt(np.pi * (1.0 + 2.0 * x ** 2 * sigma ** 2))


def example2_field(kappa, radial_F, delta=EX2["delta"], tau=TAU):
    """Latent field shared by both Example-2 circuits, parametrized by the
    radial function: rotation-invariant magnitude, angular shift delta."""
    kappa = np.asarray(kappa, dtype=float)
    r = np.linalg.norm(kappa)
    if r < 1e-12:
        return -kappa / tau
    return -kappa / tau + radial_F(r) * (rot2d(delta) @ kappa) / r


def Phi_functions(C=None, delta=EX2["delta"], J=EX2["J"], sigma=EX2["sigma"],
                  J_gaus=EX2["J_gaus"]):
    """The radial self-consistency functions Phi(varrho) of both circuits,
    in units of tau R cos(delta); the limit-cycle radius solves
    varrho = tau J cos(delta) Phi(varrho)."""
    if C is None:
        C = TAU * R_RATE * np.cos(delta)
    Phi_ring = lambda x: C * J * x / (2 * np.sqrt(np.pi)) * np.exp(-x ** 2 / 2) \
        * (i1(x ** 2 / 2) + i0(x ** 2 / 2))
    Phi_gaus = lambda x: C * J_gaus * sigma ** 2 * x / np.sqrt(np.pi * (1 + 2 * x ** 2 * sigma ** 2))
    return Phi_ring, Phi_gaus

We place $N = 20\,000$ neurons on the Gaussian line, build $F$ and $G$, and run the
rate network from four initial conditions. The same trajectories' firing rates then give
the neuronal embedding: PCA of the population activity, each neuron's loadings being its
coordinates in similarity space, colored by its true location $z_i$.

In [ ]:
# ---------------- Example 1: simulate from four initial conditions
rng = np.random.default_rng(4321)
Z1 = np.sort(rng.standard_normal(N_NEURONS))[:, None]
f_funcs, g_funcs = example1_functions()
F1, G1 = connectivity_matrices(Z1, f_funcs, g_funcs)
k_scale = example1_unit_kappa()          # latent unit k = J tau R / sqrt(2 pi)
fps1 = example1_fixed_points()
print(f"fixed points (units of k): "
      + ", ".join(f"{n}={np.round(v/k_scale, 2)}" for n, v in fps1.items()))

times1 = np.arange(0, T_SIM + DT / 2, DT)
inits1 = {"(-,-)": -1.2, "(+,+)": 1.2}
kappa_trajs1, rates1 = {}, []
for label, s in inits1.items():
    kappa0 = k_scale * s * np.ones(2)
    res = run_network(F1, G1, kappa0, times1, TAU,
                      lambda h: erf_transfer(h, R=R_RATE), device=DEVICE)
    kappa_trajs1[label] = res["kappa_t"]
    rates1.append(res["r_t"][::10])      # 1 ms sampling is enough for PCA
    print(f"start {label}: final kappa = {np.round(res['kappa_t'][-1]/k_scale, 2)} k, "
          f"nearest analytic fp "
          f"{min(fps1, key=lambda n: np.linalg.norm(res['kappa_t'][-1]-fps1[n]))}")
rates1 = np.concatenate(rates1)          # (2T, N) activity for the embedding

**The figure.** Left: the analytic latent field (speed as color, streamlines in white)
with its three stable fixed points (circles) and two saddles (crosses), and the simulated
latent trajectories on top — they run straight into the attractors. Right: the neuronal
embedding. Although the dynamics is two-dimensional, the loadings fall on a *one-*
dimensional curve, ordered by the true location $z_i$ — the footprint of the
one-dimensional circuit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
ax = axes[0]
lim = 1.8
plot_latent_field(ax, example1_field, lim)
plot_fixed_points(ax, fps1, k_scale)
for label, traj in kappa_trajs1.items():
    ax.plot(traj[:, 0] / k_scale, traj[:, 1] / k_scale, color="k", lw=1.0)
    ax.plot(traj[0, 0] / k_scale, traj[0, 1] / k_scale, "s", ms=4, color="k")
ax.set_xlabel(r"$\kappa_1$ / $k$")
ax.set_ylabel(r"$\kappa_2$ / $k$")
ax.set_title("latent vector field (1D circuit, rank 2)")

ax = axes[1]
pca1 = pca_fit(low_pass(rates1), n_components=7)
loadings1 = pca1["components"][:2].T     # (N, 2): PC1/PC2 loading per neuron
plot_embedding_2d(ax, loadings1, Z1[:, 0], cmap="RdBu_r",
                  colorbar_label="circuit location $z_i$")
ax.set_title("neuronal embedding (PC loadings)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_example1_field_and_embedding.png")
plt.show()
print(f"explained variance, PC1..3: {np.round(pca1['ratios'][:3], 3)}")

## Different circuits, identical dynamics

The second example is the degeneracy result in its purest form — two circuits that share
*every* latent trajectory, yet have different circuit spaces.

* **Ring circuit** ($d = 1$): locations uniform on $[0, 2\pi)$, connectivity
  $W_{ij} = J\cos(z_i - z_j - \delta)$ — a classic ring model with a *shift* $\delta$.
  In connectivity vectors: $f(z) = (\cos z, \sin z)$, $g(z) = J f(z + \delta)$.
* **Planar circuit** ($d = 2$): locations Gaussian in the plane with variance
  $\sigma^2$, $\sigma = 0.6$, and $W_{ij} = J'\, z_i^{T} R(\delta)\, z_j$ with $R(\delta)$
  a rotation and $J' = 1.39$.

With $\delta = \pi/10$, $J = 1$, both latent dynamics reduce to the same radial form,

$$\frac{d\kappa}{dt} = -\frac{\kappa}{\tau} + \mathcal{F}(r)\,R(\delta)\frac{\kappa}{r},
\qquad r = |\kappa|,$$

with model-specific radial functions $\mathcal F$ — Bessel functions for the ring,
$\mathcal{F}_{\mathrm{ring}}(r) = \tfrac{JRr}{2\sqrt\pi}e^{-r^2/2}\bigl[I_1(r^2/2) +
I_0(r^2/2)\bigr]$, and an algebraic one for the plane,
$\mathcal{F}_{\mathrm{gaus}}(r) = J'Rr\sigma^2/\sqrt{\pi(1+2\sigma^2 r^2)}$ — chosen so
both cross $r/\tau$ at the same radius $r_0 \approx 2.93$. The result: a globally
attractive **limit cycle** whose angular speed

$$\omega = \frac{\tan\delta}{\tau}$$

depends on neither the transfer function nor the location distribution — a pure footprint
of the connectivity shift. No measurement of latent trajectories can tell the two
circuits apart; the neuronal embeddings can (a ring vs a filled disk).

In [ ]:
# ---------------- Example 2: ring vs planar circuit
delta_ex2, r0_ex2 = EX2["delta"], EX2["r0"]
kappa0_ex2 = r0_ex2 * np.array([1.2, -1.3])
times2 = np.arange(0, T_SIM + DT / 2, DT)
ex2 = {}
for model in ("ring", "gaus"):
    Z = (np.sort(rng.uniform(0, 2 * np.pi, N_NEURONS))[:, None] if model == "ring"
         else EX2["sigma"] * rng.standard_normal((N_NEURONS, 2)))
    f_funcs, g_funcs = example2_functions(model)
    F2, G2 = connectivity_matrices(Z, f_funcs, g_funcs)
    res = run_network(F2, G2, kappa0_ex2, times2, TAU,
                      lambda h: erf_transfer(h, R=R_RATE), device=DEVICE)
    ex2[model] = dict(Z=Z, kappa=res["kappa_t"], rates=res["r_t"])

theta = np.unwrap(np.arctan2(ex2["ring"]["kappa"][:, 1], ex2["ring"]["kappa"][:, 0]))
half = len(times2) // 2
omega_num = np.polyfit(times2[half:], theta[half:], 1)[0]
print(f"angular speed on the limit cycle: measured {omega_num:.5f} rad/ms, "
      f"theory tan(delta)/tau = {np.tan(delta_ex2)/TAU:.5f} rad/ms "
      f"({abs(omega_num - np.tan(delta_ex2)/TAU)/(np.tan(delta_ex2)/TAU)*100:.1f}% off)")
for model in ("ring", "gaus"):
    r = np.linalg.norm(ex2[model]["kappa"][half:], axis=1)
    print(f"{model:5s} limit-cycle radius {r.mean():.3f} (theory {r0_ex2})")

**The figure.** Left: both simulated latent trajectories spiral onto the same circle
(dashed, radius $r_0$) and rotate at the predicted angular speed — the two circuits are
macroscopically indistinguishable. Middle and right: the neuronal embeddings, each neuron
colored by its circuit location. The ring circuit embeds as a *ring*; the planar circuit
as a *filled disk*. Same dynamics, different similarity-space geometry.

In [ ]:
fig = plt.figure(figsize=(9.6, 3.1))
ax = fig.add_subplot(1, 3, 1)
circ = r0_ex2 * np.exp(1j * np.linspace(0, 2 * np.pi, 200))
ax.plot(circ.real, circ.imag, "--", color="0.4", lw=1.0, label="theory $r_0$")
for model, c, lab in (("ring", "tab:blue", "ring circuit"),
                      ("gaus", "tab:orange", "planar circuit")):
    kk = ex2[model]["kappa"]
    ax.plot(kk[:, 0], kk[:, 1], color=c, lw=0.8, alpha=0.8, label=lab)
ax.set_aspect("equal")
ax.set_xlabel(r"$\kappa_1$")
ax.set_ylabel(r"$\kappa_2$")
ax.legend(frameon=False, loc="upper right")
ax.set_title("identical latent dynamics")

for i, model in enumerate(("ring", "gaus")):
    ax = fig.add_subplot(1, 3, i + 2)
    pca = pca_fit(low_pass(ex2[model]["rates"][::10]), n_components=7)
    Z = ex2[model]["Z"]
    color_by = Z[:, 0] if model == "ring" else np.arctan2(Z[:, 1], Z[:, 0])
    plot_embedding_2d(ax, pca["components"][:2].T, color_by, cmap="twilight",
                      colorbar_label="$z_i$ (angle)")
    ax.set_title(f"embedding: {model} circuit")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_example2_limit_cycle_embeddings.png")
plt.show()

## The task: context-dependent decision-making

The main application (Fig. 4) uses the context-dependent decision-making task of Mante et
al. (2013). The network watches streams of evidence about two visual features —
**color** and **motion** — each positive or negative on every trial, plus a **context**
cue that says which feature is relevant right now; the correct choice is the sign of the
relevant feature. Trials run back-to-back without gaps, so the network must hold the
choice until the next trial reconfigures it.

Concretely: 32 consecutive trials of 500 ms. On trial $n$ the signs alternate in a fixed
schedule (context every trial, motion every two, color every four), so the 32 trials are
four repetitions of the 8 conditions. Within a trial the context cue is on during
100–300 ms and the sensory pulses during 100–200 ms:

$$\kappa_I(t) = \bigl[I_{\mathrm{color}}(t),\, I_{\mathrm{motion}}(t),\,
I_{\mathrm{context}}(t),\, I_{\mathrm{hidden}}(t)\bigr],$$

$$I_{\mathrm{context}} = \tfrac{1.5}{\tau}\,\mathrm{context}(t),\qquad
I_{1,2} = \tfrac{0.1}{\tau}\bigl(s_{1,2}\,\mathrm{pulse}(t) + \xi_t/5\bigr),$$

with $\xi_t$ white noise — the sensory evidence is deliberately noisy, like real
recordings. $I_{\mathrm{hidden}}$ is a slow Ornstein–Uhlenbeck process
($\tau_{\mathrm{OU}} = 100$ ms) that only matters for the fifth circuit below.

In [ ]:
import json
from pathlib import Path

import numpy as np

TAU = 10.0            # ms
R_RATE = 1.0          # kHz
DT = 0.1              # ms
N_NEURONS = 3000
N_TRIALS = 32
T_TRIAL = 500.0       # ms
CONTEXT_WIN = (100.0, 300.0)   # ms within a trial
SENSE_WIN = (100.0, 200.0)     # ms within a trial
CONTEXT_STRENGTH = 1.5
INPUT_STRENGTH = 0.1
STORE_EVERY = 10      # keep one time slice out of 10 (1 ms resolution)
SEED = 1234


def cdm_inputs(n_trials=N_TRIALS, dt=DT, tau=TAU, seed=SEED):
    """Trial schedule and four-dimensional latent input kappa_I(t).

    Returns dict with times (T,), inputs (T, 4), and the per-trial sign
    vectors conts / s1 / s2 (length n_trials): context alternates every
    trial, motion sign every 2, color sign every 4, so the 32 trials cover
    the 8 conditions 4 times each.
    """
    rng = np.random.default_rng(seed)
    times = np.arange(dt, T_TRIAL * n_trials + dt / 2, dt)
    t_trial = np.arange(dt, T_TRIAL + dt / 2, dt)
    cont_win = (CONTEXT_WIN[0] < t_trial) & (t_trial < CONTEXT_WIN[1])
    sense_win = (SENSE_WIN[0] < t_trial) & (t_trial < SENSE_WIN[1])

    n = np.arange(1, n_trials + 1)
    conts = (-1.0) ** n
    s1 = (-1.0) ** ((n + 1) // 4)
    s2 = (-1.0) ** ((n + 1) // 2)

    context = np.concatenate([c * cont_win for c in conts])
    sense1 = np.concatenate([s * sense_win for s in s1])
    sense2 = np.concatenate([s * sense_win for s in s2])

    I_context = CONTEXT_STRENGTH / tau * context
    I1 = INPUT_STRENGTH / tau * (sense1 + rng.standard_normal(len(times)) / 5.0)
    I2 = INPUT_STRENGTH / tau * (sense2 + rng.standard_normal(len(times)) / 5.0)

    # hidden input: OU process with time constant 10 tau = 100 ms
    hidden = np.zeros(len(times))
    noise = rng.standard_normal(len(times)) * dt / tau / 50.0
    for t in range(1, len(times)):
        hidden[t] = hidden[t - 1] * (1.0 - dt / (10.0 * tau)) + noise[t]

    return {"times": times, "inputs": np.stack([I1, I2, I_context, hidden], 1),
            "conts": conts, "s1": s1, "s2": s2}


# --------------------------------------------------------------------- #
# The five circuit structures
# --------------------------------------------------------------------- #
def sample_locations(name, n=N_NEURONS, rng=None):
    """Draw n locations in the circuit space of one of the five models."""
    rng = rng or np.random.default_rng(SEED)
    if name in ("ring", "hid"):
        return rng.uniform(0.0, 2.0 * np.pi, size=(n, 1))
    if name == "MS":
        return rng.uniform(-1.0, 1.0, size=(n, 2))
    if name == "PS":        # cross: uniform on [-1,1]x{0} u {0}x[-1,1]
        Z = np.zeros((n, 2))
        which = rng.uniform(size=n) > 0.5
        Z[which, 0] = rng.uniform(-1.0, 1.0, which.sum())
        Z[~which, 1] = rng.uniform(-1.0, 1.0, (~which).sum())
        return Z
    if name == "clust":     # four clusters at (+-1, 0), (0, +-1)
        return np.sign(sample_locations("PS", n, rng))
    raise ValueError(name)


MODEL_PARAMS = {   # name -> (coupling J, latent scale unit_kappa, hidden K)
    "ring": dict(J=0.65, unit_kappa=1.31, K=0.0),
    "MS":   dict(J=1.00, unit_kappa=1.38, K=0.0),
    "PS":   dict(J=1.50, unit_kappa=1.33, K=0.0),
    "clust": dict(J=0.60, unit_kappa=1.30, K=0.0),
    "hid":  dict(J=0.65, unit_kappa=1.31, K=0.39),
}
MODEL_ORDER = ["ring", "MS", "PS", "clust", "hid"]


def build_cdm_network(name, n=N_NEURONS, seed=SEED):
    """Connectivity matrices F, G (N, 4) and locations Z (N, d+1) for one
    of the five CDM circuits. The last column of Z is the hidden feature
    xi = +-1, shared across models for a given seed."""
    rng = np.random.default_rng(seed)
    Z_base = sample_locations(name, n, rng)
    xi = (np.random.default_rng(seed + 1).uniform(size=n) > 0.5) * 2.0 - 1.0
    Z = np.concatenate([Z_base, xi[:, None]], axis=1)

    if name in ("ring", "hid"):
        f1, f2 = np.cos(Z[:, 0]), np.sin(Z[:, 0])
    else:
        f1, f2 = Z[:, 0], Z[:, 1]
    J, K = MODEL_PARAMS[name]["J"], MODEL_PARAMS[name]["K"]
    fI = np.abs(f1) - np.abs(f2)
    F = np.stack([f1, f2, fI, xi], axis=1)
    G = np.stack([J * f1, J * f2, np.zeros(n), K * xi], axis=1)
    return F, G, Z


def cdm_cache_path(cache_dir, name):
    return Path(cache_dir) / f"cdm_{name}.npz"


def run_or_load_cdm(name, cache_dir, task=None, device="cpu", n=N_NEURONS,
                    seed=SEED, verbose=True):
    """Load-first driver: simulate one CDM network (or reload the cached
    rates / latents / locations). run_meta.json is written last as the
    completion flag."""

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cdm_cache_path(cache_dir, name)
    meta_path = cache_dir / f"cdm_{name}_run_meta.json"
    if path.exists() and meta_path.exists():
        if verbose:
            print(f"[{name}] loaded from {path.name}")
        out = dict(np.load(path))
        out["meta"] = json.loads(meta_path.read_text())
        return out

    task = task or cdm_inputs(seed=seed)
    F, G, Z = build_cdm_network(name, n=n, seed=seed)
    if verbose:
        print(f"[{name}] simulating {len(task['times'])} steps "
              f"({N_TRIALS} trials) of N={n} neurons ...")
    phi = lambda h: erf_transfer(h, R=R_RATE, threshold=1.0)
    res = run_network(F, G, kappa0=np.zeros(4), times=task["times"], tau=TAU,
                      phi=phi, kappa_I=task["inputs"], store_every=STORE_EVERY,
                      device=device)
    np.savez_compressed(
        path, Z=Z, F=F, G=G,
        r_t=res["r_t"].astype(np.float16),
        kappa_t=res["kappa_t"], times_stored=res["times_stored"])
    meta = {"model": name, "n": n, "n_trials": N_TRIALS, "seed": seed,
            **MODEL_PARAMS[name]}
    meta_path.write_text(json.dumps(meta, indent=1))
    if verbose:
        print(f"[{name}] saved to {path.name}")
    out = dict(np.load(path))
    out["meta"] = meta
    return out

In [ ]:
# ---------------- the input schedule (first four trials shown)
task = cdm_inputs()
fig, ax = plt.subplots(figsize=(7.2, 2.2))
plot_input_timeline(ax, task["times"], task["inputs"], t_max=4 * T_TRIAL,
                    labels=["color $I_1$", "motion $I_2$",
                            "context $I_c$", "hidden $I_h$"],
                    colors=["tab:red", "tab:blue", "k", "0.6"])
for n in range(5):
    ax.axvline(n * T_TRIAL, color="0.85", lw=0.5, zorder=0)
ax.set_title("latent input channels, trials 1–4")
fig.savefig(FIG_DIR / "fig4_task_inputs.png")
plt.show()

## Five circuits solving the same task

Now the point of the paper, played out in one task. Five different circuits — a ring, a
feature plane (MS), a cross (PS), four clusters (clust), and a *double* ring (hid, two
rings linked by near-critical hidden coupling $K = 0.39$) — all solve the task with the
**same two-dimensional latent dynamics**. All five share the connectivity functions
$f_1, f_2$ of their locations (cos/sin on the rings, the coordinates themselves in the
plane variants), recurrent coupling $g = J f$, and two extra connectivity directions:

$$f = \bigl[f_1,\, f_2,\, f_I \equiv |f_1| - |f_2|,\, \xi\bigr],
\qquad g = \bigl[J f_1,\, J f_2,\, 0,\, K\xi\bigr],$$

where $\xi = \pm 1$ is a random hidden feature of each neuron. The mechanism that makes
one dynamics out of five circuits is the distinction between **aligned** and
**orthogonal** inputs:

* **Sensory inputs are aligned** with the recurrent connectivity: they enter through the
  columns $f_1, f_2$, so they push the latents linearly, $d\kappa_{1,2} \supset
  I_{1,2}\,dt$ — evidence accumulation.
* **The context input is orthogonal**: it enters through $f_I$, whose $g$-component is
  zero, so it never moves $\kappa$ directly. Instead it shifts the *argument of
  $\varphi$* and thereby reshapes the vector field — in context $+1$ the attractors sit
  on the color axis, in context $-1$ on the motion axis. Context selects *where the
  attractors are*; the aligned sensory evidence then pushes the state toward the
  corresponding one.
* **The hidden input** $I_{\mathrm{hidden}}$ enters through $\xi$ and is only amplified
  in the double-ring circuit, where $K = 0.39$ sits just below the critical coupling —
  a traceless-to-the-task latent that will later betray that circuit's topology.

| model | circuit space $\mathcal{Z}$ | $f_1, f_2$ | $J$ | hidden $K$ |
|---|---|---|---|---|
| ring | $z \sim U[0, 2\pi)$ | $\cos z,\ \sin z$ | 0.65 | — |
| MS | $z \sim U[-1,1]^2$ | $z_1,\ z_2$ | 1.0 | — |
| PS | cross $[-1,1]\times\{0\} \cup \{0\}\times[-1,1]$ | $z_1,\ z_2$ | 1.5 | — |
| clust | four points $(\pm 1, 0), (0, \pm 1)$ | $z_1,\ z_2$ | 0.6 | — |
| hid | ring $\times$ $\{\pm 1\}$ (two rings) | $\cos z,\ \sin z$ | 0.65 | 0.39 |

All networks have $N = 3000$ neurons, $\tau = 10$ ms, simulated at $dt = 0.1$ ms with the
threshold-shifted transfer function $\varphi(h) = \tfrac{R}{2}\bigl(\mathrm{erf}(h-1) +
1\bigr)$.

In [ ]:
# ---------------- the five circuit spaces
fig, axes = plt.subplots(1, 5, figsize=(10.5, 2.2))
for ax, name in zip(axes, MODEL_ORDER):
    F, G, Z = build_cdm_network(name)
    if name in ("ring", "hid"):
        ax.scatter(np.cos(Z[:, 0]), np.sin(Z[:, 0]),
                   c=Z[:, 1] if name == "hid" else Z[:, 0],
                   cmap="tab10" if name == "hid" else "twilight", s=1)
    elif name == "clust":        # four clusters: one color per cluster
        ax.scatter(Z[:, 0], Z[:, 1], c=Z[:, 0] + 2 * Z[:, 1], cmap="tab10", s=3)
    elif name == "PS":           # cross: position along either arm
        ax.scatter(Z[:, 0], Z[:, 1], c=Z[:, 0] + Z[:, 1], cmap="RdBu_r", s=1)
    else:                        # MS: feature plane, colored by z_2
        ax.scatter(Z[:, 0], Z[:, 1], c=Z[:, 1], cmap="RdBu_r", s=1)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(name, fontsize=8)
fig.suptitle("the five circuit spaces (one point per neuron)", y=1.02)
fig.savefig(FIG_DIR / "fig4_circuit_spaces.png")
plt.show()

## Simulation: all five solve the task

We simulate each circuit over the full 32-trial session (16 s of activity) and read the
choice off the population by projecting onto the $f_1 + f_2$ direction — the axis along
which both features' evidence is integrated; the choice on each trial is the sign of the
readout at its most extreme excursion. The simulations are cached under
`models/manifold_and_circuit/`; rerunning this notebook reloads them.

In [ ]:
# ---------------- run or load the five CDM simulations
cdm = {}
for name in MODEL_ORDER:
    cdm[name] = run_or_load_cdm(name, MODEL_DIR, task=task, device=DEVICE)

print(f"{'model':6s} {'choice accuracy':>15s} {'std(kappa_hidden)':>18s}")
for name in MODEL_ORDER:
    out = cdm[name]
    rates_sm = low_pass(out["r_t"].astype(np.float64))
    axes_ro = readout_axes(out["F"], out["meta"]["unit_kappa"])
    choices = trial_choices(rates_sm @ axes_ro["choice"], out["times_stored"], T_TRIAL)
    acc = (choices == correct_choices(task)).mean()
    cdm[name]["rates_sm"] = rates_sm
    cdm[name]["accuracy"] = acc
    print(f"{name:6s} {acc:15.3f} {out['kappa_t'][:, 3].std():18.4f}")

Every circuit solves the task (the double ring slightly less cleanly — its
near-critical hidden coupling leaks noise into the dynamics). Note the last column: the
hidden latent $\kappa_{\mathrm{hidden}}$ fluctuates an order of magnitude more in `hid`
than anywhere else. The task doesn't need it; the *critical amplification* of the hidden
OU input produces it. That fluctuation is what will make the double ring visible in the
embedding below.

**Example trials.** The choice readout over the first eight trials of the ring circuit,
green where the readout's sign matches the correct choice:

In [ ]:
out = cdm["ring"]
axes_ro = readout_axes(out["F"], out["meta"]["unit_kappa"])
choice_trace = out["rates_sm"] @ axes_ro["choice"]
fig, ax = plt.subplots(figsize=(7.2, 2.4))
plot_choice_traces(ax, out["times_stored"], choice_trace, task, n_trials_show=8)
ax.set_title(f"ring circuit: choice readout (accuracy {out['accuracy']:.2f})")
fig.savefig(FIG_DIR / "fig4_choice_readout.png")
plt.show()

## Same latent dynamics

**Context reshapes the field.** Clamp the context latent at its steady-state value
$\kappa_{\mathrm{context}} = 1.5\,c$ and draw the latent field in the $(\kappa_{\mathrm{
color}}, \kappa_{\mathrm{motion}})$ plane for the ring circuit: in context $+1$ the two
attractors sit on the *color* axis, in context $-1$ on the *motion* axis. This is the
orthogonal-input mechanism in action — context never pushes the state, it moves the
attractors.

In [ ]:
# ---------------- context-dependent latent fields (ring circuit)
F_r, G_r = cdm["ring"]["F"], cdm["ring"]["G"]
phi_cdm = lambda h: erf_transfer(h, R=R_RATE, threshold=1.0)

def clamped_field(kappa2, context):
    """Latent field in the (color, motion) plane with the context latent
    clamped at its steady-state value 1.5 * context."""
    k = np.array([kappa2[0], kappa2[1], CONTEXT_STRENGTH * context, 0.0])
    drive = (G_r.T @ phi_cdm(torch.as_tensor(F_r @ k, dtype=torch.float32)).numpy()
             / len(F_r))
    return -kappa2 / TAU + drive[:2]

fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.1))
for ax, ctx in zip(axes, (+1, -1)):
    plot_latent_field(ax, clamped_field, lim=2.2, n_grid=22, extra_args=(ctx,))
    ax.set_xlabel(r"$\kappa_{\mathrm{color}}$")
    ax.set_title(f"context {ctx:+d}: attractors on the "
                 + ("color" if ctx > 0 else "motion") + " axis")
axes[0].set_ylabel(r"$\kappa_{\mathrm{motion}}$")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_context_fields.png")
plt.show()

**All five circuits trace the same trajectories.** Targeted dimensionality reduction
(TDR, Mante et al. 2013) extracts task axes from each network: denoise the activity with
its first 7 PCs, regress the trial-concatenated activity onto the design
$[\mathrm{color},\ \mathrm{motion},\ \mathrm{context},\ 1]$, take each regressor's
coefficient vector across neurons at the time of its maximal norm, and QR-orthogonalize —
giving orthonormal color, motion, and context axes per network. Projecting the
block-averaged activity (four repetitions of the 8-condition block) onto the color and
motion axes shows what the task requires: in color context, trajectories split along the
color axis; in motion context, along the motion axis. And the five circuits' trajectories
lie on top of each other — same latent dynamics.

In [ ]:
# ---------------- TDR axes and block-averaged trajectories per model
model_colors = {"ring": "tab:blue", "MS": "tab:orange", "PS": "tab:green",
                "clust": "tab:red", "hid": "tab:purple"}
trajs_avg = {}
for name in MODEL_ORDER:
    out = cdm[name]
    pca = pca_fit(out["rates_sm"], n_components=7)
    denoised = pca_denoise(pca, out["rates_sm"])
    dt_stored = out["times_stored"][1] - out["times_stored"][0]
    axes_tdr = tdr_axes(denoised, out["rates_sm"], task, T_TRIAL, dt_stored)
    trajs = (axes_tdr.T @ out["rates_sm"].T)[:2]         # (color, motion) x T
    avg = block_average(trajs, n_reps=4)                 # average over 4 reps
    steps_block = avg.shape[-1]
    steps_trial = int(round(T_TRIAL / dt_stored))
    trajs_avg[name] = avg.reshape(2, 8, steps_trial).transpose(1, 0, 2)  # (8, 2, T)

fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.2))
for ax, ctx in zip(axes, (+1.0, -1.0)):
    plot_tdr_panel(ax, trajs_avg, task, dt_stored, ctx, model_colors, MODEL_ORDER)
    ax.set_title(f"context {ctx:+.0f}: "
                 + ("color" if ctx > 0 else "motion") + " is relevant")
handles = [plt.Line2D([], [], color=model_colors[n], label=n) for n in MODEL_ORDER]
axes[1].legend(handles=handles, frameon=False, loc="upper left",
               bbox_to_anchor=(1.02, 1.0))
fig.suptitle("TDR trajectories, all five circuits overlaid", y=1.0)
fig.savefig(FIG_DIR / "fig4_tdr_trajectories.png")
plt.show()

## Different neuronal embeddings

The latent trajectories are identical across circuits; the **neuronal embeddings** are
not. Run PCA on the full session's activity (16 s, low-passed), and each neuron's
first loadings are its coordinates in similarity space. Because activity is a smooth
function of circuit location, the embedding point cloud is an image of the circuit space:
a ring stays a ring, a plane stays a patch, a cross stays a cross, four clusters stay four
points — and the double ring, lifted apart by its hidden-feature latent, appears as **two
rings**. (For the double ring the hidden feature straddles PCs 4–5, so the third shown
coordinate is the direction that best separates the $\xi = \pm 1$ neurons.) Points are
colored by their coordinate on the third shown axis.

In [ ]:
# ---------------- PCA embeddings for the five models (full session)
show_dims = {"ring": (0, 1, 2), "MS": (0, 1, 2), "PS": (0, 1, 2),
             "clust": (0, 1, 2), "hid": (0, 1, 3)}   # hid: PC4 carries xi
model_cmaps = {"ring": "Blues", "MS": "Oranges", "PS": "Greens",
               "clust": "Reds", "hid": "Purples"}
fig = plt.figure(figsize=(11.5, 2.9))
for i, name in enumerate(MODEL_ORDER):
    out = cdm[name]
    pca = pca_fit(out["rates_sm"], n_components=7)
    loadings = pca["components"].T                      # (N, 7)
    dims = show_dims[name]
    emb = loadings[:, dims].copy()
    if name == "hid":
        # the hidden feature direction in loading space: mean difference
        # between the xi = +1 and xi = -1 neurons (it straddles PC4/PC5)
        xi = out["Z"][:, 1]
        xi_axis = loadings[xi > 0].mean(0) - loadings[xi < 0].mean(0)
        xi_axis /= np.linalg.norm(xi_axis)
        emb[:, 2] = loadings @ xi_axis
    ax = fig.add_subplot(1, 5, i + 1, projection="3d")
    view = dict(elev=8, azim=-75) if name == "hid" else dict()
    plot_embedding_3d(ax, emb, emb[:, 2], cmap=model_cmaps[name], s=3, **view)
    ax.set_title(name, fontsize=8)
    cdm[name]["pca"] = pca
fig.suptitle("neuronal embeddings (PCA loadings, full session)", y=1.04)
fig.savefig(FIG_DIR / "fig4_embeddings.png")
plt.show()

## The discrimination table

Now the payoff. Suppose an experiment hands you the neuronal embedding of a real
decision-making circuit, and the five model families above are the candidates. The
topological footprint gives a hard rule: a continuous map cannot raise intrinsic
dimension or merge disconnected components, so **the embedding cannot have higher
intrinsic dimension or more connected components than the circuit space that generated
it**. Checking each (data, candidate) pair against the rule:

* ring data (a circle: $d=1$, one component) rules out only the discrete four-cluster
  space ($d=0$);
* MS data (a 2D patch) rules out every 1D circuit — ring, cross, and double ring alike;
* clust data (four points) rules out every connected circuit;
* hid data (two rings) rules out every simply-connected one — and every 0D one.

Each data row excludes a *different* set of candidates: the five circuits, identical in
their latent dynamics, are all told apart by the topology of their neuronal embeddings.

In [ ]:
excluded = discrimination_matrix(MODEL_ORDER)
print("excluded (row = data, column = candidate):")
print("        " + "  ".join(f"{n:>5s}" for n in MODEL_ORDER))
for i, name in enumerate(MODEL_ORDER):
    print(f"{name:6s} " + "  ".join("    X" if excluded[i, j] else "    ."
                                   for j in range(len(MODEL_ORDER))))

fig, ax = plt.subplots(figsize=(3.6, 3.4))
plot_discrimination_table(ax, MODEL_ORDER, excluded)
ax.set_title("topological footprints discriminate what dynamics cannot")
fig.savefig(FIG_DIR / "fig4_discrimination_table.png")
plt.show()

## Summary

* **Degeneracy is real.** A one-dimensional circuit can host two-dimensional dynamics
  (Example 1), and microscopically different circuits — a ring and a plane — produce
  *identical* latent dynamics, down to the limit cycle and its angular speed
  $\omega = \tan\delta/\tau$ (Example 2). Latent trajectories alone cannot identify a
  circuit.
* **But embeddings leave footprints.** Neurons embedded by their activity form a point
  cloud that is a continuous image of the circuit space. Symmetries of the circuit show
  up as symmetries of the dynamics, and the topology of the embedding — its intrinsic
  dimension and number of connected components — bounds that of the circuit from below.
* **Applied to a cognitive task**, five circuits solve context-dependent decision-making
  with the same latent dynamics (same context-dependent fields, same TDR trajectories,
  perfect choices), yet their single-trial PCA embeddings — ring, patch, cross, four
  clusters, *two* rings — are all topologically distinct, and the discrimination rule
  separates all five.

*Choices made here*: networks are simulated as deterministic rate models (the large-$N$
limit; input noise keeps single-trial statistics realistic) rather than Poisson spiking
networks; Example 1–2 use $N = 20\,000$ so the sampled field sits close to the large-$N$
theory (with $N = 2000$ the finite-sample field visibly warps the limit cycle, a
finite-size effect, not a discrepancy); PCA uses `neuralrnn.analysis.fit_pca`; scope is
the main-text Figures 3 and 4.